# AI活用のためのデータベース準備 ハンズオン（Section 08）

このノートブックでは、OpenAI API の準備から、  
Python で呼び出し、共通関数として整理する流れを体験します。

---

## ここでやること
- 前の章で作成したCSVファイルを読み込む
- SQLiteデータベースを作成する
- テーブル（meetings, speeches / qa）を作成する
- 発言データをデータベースに登録する
- 登録されたデータを確認する
- 次の章で使うための土台を整える

In [1]:
########################################
# テーブル作成
########################################
import sqlite3

conn = None

try:
    conn = sqlite3.connect("ank.db")
    cursor = conn.cursor()

    print("データベース 'ank.db' を作成しました。")

    # -----------------------------
    # meetings テーブル
    # -----------------------------
    sql_create_meetings_table = """
    CREATE TABLE IF NOT EXISTS meetings (
        issueID          TEXT PRIMARY KEY,
        nameOfHouse      TEXT,
        nameOfMeeting    TEXT,
        issue            TEXT,
        date             TEXT
    );
    """
    cursor.execute(sql_create_meetings_table)
    print("テーブル 'meetings' を作成しました。")

    # -----------------------------
    # speeches テーブル
    # -----------------------------
    sql_create_speeches_table = """
    CREATE TABLE IF NOT EXISTS speeches (
        speechID         TEXT PRIMARY KEY,
        issueID          TEXT NOT NULL,
        speechOrder      INTEGER,
        speaker          TEXT,
        speech           TEXT NOT NULL
    );
    """
    cursor.execute(sql_create_speeches_table)
    print("テーブル 'speeches' を作成しました。")

    # -----------------------------
    # qa テーブル
    # -----------------------------
    sql_create_qa_table = """
    CREATE TABLE IF NOT EXISTS qa (
        qa_id            INTEGER PRIMARY KEY AUTOINCREMENT,
        speechID         TEXT NOT NULL,
        question         TEXT NOT NULL,
        answer           TEXT NOT NULL,
        created_at       TEXT DEFAULT CURRENT_TIMESTAMP
    );
    """
    cursor.execute(sql_create_qa_table)
    print("テーブル 'qa' を作成しました。")

    conn.commit()
    print("テーブル作成を保存しました。")

except sqlite3.Error as e:
    print(f"データベース操作中にエラーが発生しました: {e}")
    if conn:
        conn.rollback()

except Exception as e:
    print(f"予期しないエラーが発生しました: {e}")

finally:
    if conn:
        conn.close()
        print("データベース接続をクローズしました。")

データベース 'ank.db' を作成しました。
テーブル 'meetings' を作成しました。
テーブル 'speeches' を作成しました。
テーブル 'qa' を作成しました。
テーブル作成を保存しました。
データベース接続をクローズしました。


In [ ]:
import requests
import sqlite3
import pandas as pd
from google.colab import files

# ==============================
# DB 接続
# ==============================
conn = sqlite3.connect("ank.db")
cursor = conn.cursor()

# ==============================
# 国会議事録 API 取得
# ==============================
all_records = []

for start in range(1, 1001, 10):
    params = {
        "recordPacking": "json",
        "any": "数学",
        "from": "2022-01-01",
        "until": "2025-12-31",
        "maximumRecords": 10,
        "startRecord": start
    }

    BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
    except requests.exceptions.HTTPError:
        print(f"取得終了（startRecord={start}）")
        break

    data = response.json()
    records = data.get("speechRecord", [])
    all_records.extend(records)

print(f"取得件数: {len(all_records)}")

if len(all_records) == 0:
    print("データが取得できませんでした。")
else:
    # ==============================
    # DataFrame（確認用）
    # ==============================
    df = pd.DataFrame([
        {
            "speech_id": r.get("speechID"),
            "date": r.get("date"),
            "meeting": r.get("nameOfMeeting"),
            "speaker": r.get("speaker"),
            "committee": r.get("speakerGroup"),
            "speech_text": r.get("speech", "")
        }
        for r in all_records
    ])

    display(df.head(10))

    # ==============================
    # DB 登録（重複はスキップ）
    # ==============================
    inserted = 0
    skipped = 0

    for _, row in df.iterrows():
        try:
            cursor.execute(
                """
                INSERT INTO speeches (
                    speech_id,
                    meeting_id,
                    date,
                    committee,
                    speaker,
                    speech_text
                )
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    row["speech_id"],
                    row["meeting"],
                    row["date"],
                    row["committee"],
                    row["speaker"],
                    row["speech_text"]
                )
            )
            inserted += 1

        except sqlite3.IntegrityError:
            # PRIMARY KEY 重複
            skipped += 1

    conn.commit()
    conn.close()

    print(f"DB 登録完了：追加 {inserted} 件 / スキップ {skipped} 件")

取得終了（startRecord=341）
取得件数: 336


,speech_id,date,meeting,speaker,committee,speech_text
0,121905206X00520251203_007,2025-12-03,法務委員会,斎藤由則,None,○斎藤参考人 初めまして。司法受験生の斎藤由則と申します。\r\n 本日は、法務委員会にお招...
1,121905124X00320251126_014,2025-11-26,文部科学委員会,青山大人,立憲民主党・無所属,○青山委員 大臣の、この一年間取り組んでこられた思いを是非聞きたかったんですけれども、多分も...
2,121915104X00220251120_005,2025-11-20,文教科学委員会,望月禎,None,○政府参考人（望月禎君） お答え申し上げます。\r\n 今御指摘のとおり、文部科学省では、子...
3,121915104X00220251120_009,2025-11-20,文教科学委員会,望月禎,None,○政府参考人（望月禎君） 先ほど来申し上げていますが、いわゆるデジタル教科書につきましては、...
4,121905261X00420251111_218,2025-11-11,予算委員会,緒方林太郎,有志の会,○緒方委員 それでは、もう少し具体的に経済政策についてお伺いしたいと思います。\r\n 債務...
5,121715104X01520250620_000,2025-06-20,文教科学委員会,会議録情報,None,令和七年六月二十日（金曜日）\r\n 午後六時十八分開会\r\n ────────...
6,121705124X01720250618_000,2025-06-18,文部科学委員会,会議録情報,None,令和七年六月十八日（水曜日）\r\n 午前九時開議\r\n 出席委員\r\n 委員...
7,121705124X01720250618_119,2025-06-18,文部科学委員会,あべ俊子,自由民主党・無所属の会,○あべ国務大臣 西岡委員にお答えさせていただきます。\r\n 現行の学習指導要領に基づく学習...
8,121705124X01620250613_000,2025-06-13,文部科学委員会,会議録情報,None,令和七年六月十三日（金曜日）\r\n 午前九時開議\r\n 出席委員\r\n 委員...
9,121705124X01620250613_004,2025-06-13,文部科学委員会,大森直樹,None,○大森参考人 私からは、教育課程基準の問題点と改訂の課題について、カリキュラムオーバーロード...


DB 登録完了：追加 0 件 / スキップ 336 件
